In [ ]:
import numpy as np
import bisect
import matplotlib.pyplot as plt

import essrs

In [ ]:
print(help(essrs.model.toy.rosenbrock))

In [ ]:
TRUE_A: float = 1.0
TRUE_B: float = 3.0

In [ ]:
def enclosed_probability(sigma: int, df: int) -> float:
    return 1 - np.exp(-(sigma**2) / df)

SIGMA_1 = enclosed_probability(1, 2)
SIGMA_2 = enclosed_probability(2, 2)
SIGMA_3 = enclosed_probability(3, 2)

print(SIGMA_1)
print(SIGMA_2)
print(SIGMA_3)

In [ ]:
N_WALKERS: int = 50
N_PARAMETERS: int = 2

ics = np.random.normal(loc=[1, 1], scale=2., size=(N_WALKERS, N_PARAMETERS))

model = essrs.model.toy.rosenbrock(a=1, b=3)
burn_move = essrs.moves.hybrid_ddc_move(one_in_n=10)
sampler_move = essrs.moves.differential_move()
config = essrs.EnsembleConfig(n_steps=3000, n_walkers=N_WALKERS, n_parameters=N_PARAMETERS, n_burn_in = 2000)

fit, _ = essrs.run_mcmc(
    log_likelihood=model,
    config=config,
    burn_in_move=burn_move,
    sampling_move=sampler_move,
    initial_conditions=ics
)

In [ ]:
print(fit.shape)

x_fit = fit[:,:,0].ravel()
y_fit = fit[:,:,1].ravel()

In [ ]:
def rosenbrock(x: float, y: float, a: float = TRUE_A, b: float = TRUE_B) -> float:
    return -0.5 * (
        (a - x)**2 + b * (y - x**2)**2
    )

In [ ]:
xc = np.linspace(-2.5, 4.5, 1000)
yc = np.linspace(-2, 17, 1001)

dx = xc[1] - xc[0]
dy = yc[1] - yc[0]

gx, gy = np.meshgrid(xc, yc)
gz = np.exp(rosenbrock(gx, gy))

gz /= np.sum(gz) * (dx * dy)


In [ ]:
def estimate_enclosed_region(Z, target_sum):
    z = np.sort(Z.ravel(), descending=True)
    c = np.cumsum(z)
    i = bisect.bisect(c, target_sum)

    M = np.zeros_like(Z)
    M[z[i] < Z] = 1

    return M

In [ ]:
M1 = estimate_enclosed_region(gz*dx*dy, SIGMA_1)
M2 = estimate_enclosed_region(gz*dx*dy, SIGMA_2)
M3 = estimate_enclosed_region(gz*dx*dy, SIGMA_3)

# generate the mask array highlighting the different regions in parameter space
M = M1 + M2 + M3

In [ ]:
plt.figure()
plt.contour(gx, gy, M, cmap="spring")
plt.scatter(x_fit, y_fit, marker=".", s=0.1, color="k", alpha=0.1)
plt.xlabel(r"$x$")
plt.ylabel(r"$y$")
plt.show()

In [ ]:
z_marg_y = np.sum(gz, axis=0)
z_marg_y /= np.sum(z_marg_y) * dx
plt.figure()
plt.plot(gx[0,:], z_marg_y, color="r")
plt.hist(x_fit, density=True, histtype="step", bins=100, color="k")
plt.xlabel(r"$x$")
plt.ylabel(r"$p$")
plt.show()

In [ ]:
z_marg_x = np.sum(gz, axis=1)
z_marg_x /= np.sum(z_marg_x) * dy
plt.figure()
plt.plot(gy[:,0], z_marg_x, color="r")
plt.hist(y_fit, density=True, histtype="step", bins=100, color="k")
plt.xlabel(r"$y$")
plt.ylabel(r"$p$")
plt.show()